[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pflahert/hd-stats-workshop/blob/main/notebooks/homework.ipynb)

# Homework: Day 1 → Day 2
**Due at the start of Day 2** | Estimated time: 3–3.5 hours

This homework consolidates Day 1 material and bridges to Day 2. It maps onto the Day-1 structure:

- **Q1 — Multiple Testing Concepts (LO1, LO3)** — five sub-parts **Q1.1–Q1.5**. Exercises **Lecture 1** (hypothesis testing: Bonferroni / FWER, BH / FDR, $q$-values) and includes an **AI explanation + critique** exercise (Q1.5).
- **Q2 — FDR in Practice (LO1, LO3)**. Connects both Day-1 lectures: BH-adjusted $p$-values from **Lecture 1**; empirical-Bayes variance shrinkage (the moderated $t$-statistic, the idea behind `limma`'s `eBayes`) and Storey's $\hat\pi_0$ from **Lecture 2**. **All three sub-parts ask you to use AI for code generation and to critique what comes back against an explicit checklist.** You will implement a Smyth-style moderated-$t$ statistic *from scratch in Python* (no R, no rpy2) in roughly 15 lines of numpy + scipy.
- **Q3 — Evaluating Analysis Output, and a Day-2 bridge (LO3)** — **five sub-parts**, three of which include you-vs-AI comparisons: (a) writing specifications; (b) verifying a citation; (c) debugging code; (d) a true **in-context multi-turn** code review; (e) a short PCA-on-Golub preview that primes Lecture 3 (dimension reduction) and Lecture 4 (penalized regression).

---

**How this notebook uses AI**

This homework integrates AI at several steps, not just at the end. Two kinds of AI exercises appear:

- **Single-shot critiques** — you prompt, paste the response, and evaluate it against an explicit checklist.
- **Multi-turn conversations** — you prompt, the AI responds, you write a follow-up prompt **in the same conversation** that references the prior response, the AI revises, and you write a final assessment. *At least one true multi-turn exercise per notebook is required.* In this homework, Q3(d) is the multi-turn exercise — Turns 2 and 3 must be sent in the same chat session so the AI can see and revise its earlier answer.

Use any AI assistant (the [UMass campus GenAI platform](https://www.umass.edu/it/genai/platform), Claude, ChatGPT, or another). Note which AI you used for each prompt — different models behave differently, and the differences are part of the lesson.

---

> **This notebook uses the Python (`hd-stats`) kernel** — the same kernel as Labs 1–4. No R, no Bioconductor, no kernel switching: the entire workshop runs on one Python stack. In Colab: **Runtime → Change runtime type → Python 3** if your session is still on a stale R kernel from a prior homework version.


In [ ]:
# Verify you are running the Python (hd-stats) kernel — not the R kernel from a prior homework version.
import sys
assert sys.version_info[0] == 3, (
    "This notebook requires Python 3. If you see this from an R runtime, "
    "switch via Runtime → Change runtime type → Python 3 in Colab."
)
print(f"Python {sys.version.split()[0]}")
import os
print(f"Working directory: {os.getcwd()}")


---
## Q1: Multiple Testing Concepts (LO1, LO3)

A proteomics experiment measures the abundance of $p = 5{,}000$ proteins in $n = 30$ tissue samples (15 tumor, 15 normal). For each protein, a two-sample $t$-test is performed.

### Q1.1
Under the global null (no protein is differentially abundant), how many proteins would you expect to reject at $\alpha = 0.05$ without any correction?

**YOUR ANSWER:**


### Q1.2
The Bonferroni-corrected threshold is $\alpha / m$. Compute this threshold. A protein with a "real" effect has a $p$-value of $3 \times 10^{-4}$. Would it survive Bonferroni correction?

**YOUR ANSWER:**


In [ ]:
# You can verify Q1.1 and Q1.2 here:
m = 5000
alpha = 0.05
expected_fp_uncorrected = m * alpha
bonf_threshold = alpha / m
print(f"Expected false positives (uncorrected): {expected_fp_uncorrected:.0f}")
print(f"Bonferroni threshold: {bonf_threshold:.2e}")
print(f"Real effect p = 3e-4 survives Bonferroni? {3e-4 < bonf_threshold}")


### Q1.3
You apply the BH procedure at FDR $= 0.10$ and get 120 rejections. Give an interpretation of this result in plain language that a biologist would understand.

**YOUR ANSWER:**


### Q1.4
Your collaborator says "We found 120 significant proteins." A reviewer replies "But 12 of those are probably false positives!" Is the reviewer being unreasonable? Explain.

*Rubric note: BH controls the FDR* in expectation *over the procedure — the realized number of false discoveries in any one experiment will fluctuate around $\mathrm{FDR}\cdot R = 12$. Either "the reviewer's number is the right order of magnitude" or "the reviewer is treating an expectation as a per-experiment guarantee" can earn full credit, provided the distinction is named.*

**YOUR ANSWER:**


### Q1.5 — AI explanation + critique

Ask an AI assistant ([UMass campus GenAI platform](https://www.umass.edu/it/genai/platform), Claude, or ChatGPT) the following question and paste its response below:

> "Reply in three short paragraphs only (no .docx file, no code). Explain the difference between FWER (Bonferroni-style) and FDR (Benjamini–Hochberg-style) control to a wet-lab biologist who has never taken a statistics course. Use the analogy of finding stars in a noisy telescope image."

**The AI you used (model name):**


**AI's response (paste verbatim):**


**Your critique** — evaluate the AI's response against these criteria:

| Criterion | Pass / Fail / Partial | Notes |
|---|---|---|
| Does the AI define both FWER and FDR correctly? | | |
| Does it explain the key difference (any vs. fraction)? | | |
| Does it pick the right context (FDR for genomic discovery)? | | |
| Does it avoid the common error of saying "FDR is the probability that a finding is false"? | | |
| Does it distinguish FDR (expectation over the procedure) from the *realized* per-experiment fraction (the issue Q1.4 raises)? | | |
| Is the star/telescope analogy accurate? | | |

**One-sentence overall assessment** — would you use this explanation when introducing the concept to a wet-lab collaborator?


---
## Q2: FDR in Practice (LO1, LO3)

Using the Golub dataset (introduced in **Lecture 1** and used in **Labs 1 and 2**), perform the following analysis. You should use AI to write the Python code, but evaluate its correctness against the explicit checklist provided for each sub-part, and interpret the results yourself.

The two empirical-Bayes ideas you will exercise here:

- **From Lecture 1**: BH adjustment converts $p$-values to FDR-controlled discoveries. Storey's $\hat\pi_0$ from the same lecture provides an adaptive variant.
- **From Lecture 2**: empirical-Bayes shrinkage of gene-level variance estimates toward a common prior (the idea behind `limma`'s `eBayes()`), producing moderated $t$-statistics that are more stable than raw $t$-tests when $n$ is small.

In this homework you will implement a **Smyth-style** moderated-$t$ idea *from scratch in Python* — no R, no rpy2. The implementation is roughly 15 lines of numpy + scipy. The full Smyth procedure ([Smyth 2004](https://doi.org/10.2202/1544-6115.1027)) estimates the prior hyperparameters $(d_0, s_0^2)$ by trigamma matching; the geometric-mean choice below is a pedagogical simplification that gives qualitatively similar shrinkage.

### Setup — load the Golub data and the standard scientific-Python stack

In [ ]:
# Q2 uses the same standard scientific-Python stack as the labs:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests

# Workshop convention: deterministic seed for any AI-generated bootstrap / resampling code.
np.random.seed(2026)

import scipy, statsmodels
print(f"numpy {np.__version__}, pandas {pd.__version__}")
print(f"scipy {scipy.__version__}, statsmodels {statsmodels.__version__}")


In [ ]:
# Load the Golub expression matrix and metadata directly from the workshop repo.
# URLs are pinned to `main`; the shape assertions below will catch any silent
# upstream change before the rest of the homework runs.
expr_url = "https://raw.githubusercontent.com/pflahert/hd-stats-workshop/v2026-spring-data/data/golub_expression.csv"
meta_url = "https://raw.githubusercontent.com/pflahert/hd-stats-workshop/v2026-spring-data/data/golub_metadata.csv"

expr = pd.read_csv(expr_url, index_col=0)            # 3,051 genes (rows) x 72 samples (cols)
meta = pd.read_csv(meta_url)                          # 72 rows: sample_id, subtype

assert expr.shape == (3051, 72), f"Unexpected expression shape: {expr.shape} (expected (3051, 72))"
assert meta.shape[0] == 72, f"Unexpected metadata row count: {meta.shape[0]} (expected 72)"
assert set(meta["subtype"].unique()) == {"ALL", "AML"}, f"Unexpected subtypes: {meta['subtype'].unique()}"

print(f"Expression matrix: {expr.shape[0]} genes x {expr.shape[1]} samples")
print(f"Metadata rows: {meta.shape[0]}")
print("Class distribution:")
print(meta["subtype"].value_counts())

# Sanity check: columns of expr should match meta["sample_id"]
assert list(expr.columns) == list(meta["sample_id"]), "Sample IDs do not match."

# Group masks (boolean, length 72)
all_mask = (meta["subtype"].values == "ALL")
aml_mask = (meta["subtype"].values == "AML")
gene_names = expr.index.values
n_all = int(all_mask.sum())
n_aml = int(aml_mask.sum())
print(f"ALL: {n_all} samples; AML: {n_aml} samples")


### Q2.a — BH-adjusted p-values via hand-rolled moderated $t$-statistics

Direct an AI assistant to implement a Smyth-style moderated $t$-statistic *from scratch in Python* (i.e., not by calling an R library), then BH-adjust the resulting moderated $p$-values. Report the number of genes significant at FDR $< 0.01$, $< 0.05$, $< 0.10$.

The moderated $t$-statistic is

$$\tilde t_g = \frac{\bar y_{g,A} - \bar y_{g,B}}{\tilde s_g\, \sqrt{1/n_A + 1/n_B}},
\qquad
\tilde s_g^2 = \frac{d_0\, s_0^2 + d_g\, s_g^2}{d_0 + d_g}$$

where $s_g^2$ is the per-gene pooled sample variance, $d_g = n_A + n_B - 2$ is its degrees of freedom, and $(d_0, s_0^2)$ are prior hyperparameters fixed by a simple heuristic: $d_0 = 4$ and $s_0^2 = \exp(\text{mean of }\log s_g^2)$. The reference distribution under the null is a $t$ with $d_0 + d_g$ degrees of freedom. (Smyth's full procedure fits $(d_0, s_0^2)$ by trigamma matching; the geometric-mean choice here is a pedagogical simplification.)

**Suggested prompt** (paste-ready, with format directive):

> "Reply with a single Python code block (no prose, no .docx file — just runnable code I can paste into a Colab cell). Given pandas DataFrames `expr` (rows = 3,051 genes; columns = 72 samples; column order matches `meta['sample_id']`) and `meta` (columns `sample_id` and `subtype` ∈ {'ALL', 'AML'}), implement a Smyth-style moderated $t$-statistic from scratch: compute per-gene pooled sample variance $s_g^2$ with degrees of freedom $d_g = n_A + n_B - 2$ (note rows are genes, columns are samples — split by `meta['subtype']` to form the two groups), fix the prior hyperparameters $d_0 = 4$ and $s_0^2 = \exp(\text{mean log }s_g^2)$, compute moderated variances $\tilde s_g^2 = (d_0 s_0^2 + d_g s_g^2)/(d_0 + d_g)$ and moderated $t$-statistics, get **two-sided** p-values from a $t$-distribution with $d_0 + d_g$ degrees of freedom, then BH-adjust via `statsmodels.stats.multitest.multipletests(..., method='fdr_bh')`. Print the counts of genes with adjusted $p$-value below 0.01, 0.05, and 0.10. Use numpy, scipy.stats, and statsmodels only — no R, no rpy2."

**Your prompt to the AI:**


**The AI you used:** (e.g., the [UMass campus GenAI platform](https://www.umass.edu/it/genai/platform), ChatGPT, Claude)


**Did the AI produce the moderated-$t$ implementation in one shot, or did you have to direct it?**


**Critique checklist (complete after pasting the AI's code):**

| Criterion | Pass / Fail / Partial | Notes |
|---|---|---|
| Did the AI orient the matrix correctly (rows = genes, columns = samples)? | | |
| Did the AI use $d_0 + d_g$ degrees of freedom for the reference $t$ (not just $d_g$)? | | |
| Did the AI use a **two-sided** p-value? | | |
| Did the AI call `multipletests(method='fdr_bh')` rather than another method (e.g. `bonferroni`, `fdr_by`)? | | |
| Did the AI use $s_0^2 = \exp(\overline{\log s_g^2})$ (*geometric* mean of variances), not $\overline{s_g^2}$ (arithmetic mean)? | | |
| Did the AI compute two-sided p-values with `2 * stats.t.sf(|t|, df=d_0+d_g)` (rather than `1 - cdf`, which can underflow to 0 for large \|t\|)? | | |
| Did the AI add any silent fallback (e.g. clipping zero-variance genes) that could distort the test? | | |

In [ ]:
# Paste / refine the AI-generated moderated-t code here.
#
# Implementation hints (do NOT copy-paste a scaffold from elsewhere — let the AI
# write it, then critique against the checklist above):
#
#   * Split with `all_mask` / `aml_mask` from the load cell.
#   * d_g = n_A + n_B - 2 = 70 on Golub.
#   * Use s_0_sq = exp(mean(log(s_g_sq)))  -- the GEOMETRIC mean of variances,
#     not the arithmetic mean.
#   * For two-sided p-values, prefer `2 * stats.t.sf(np.abs(t_mod), df=d_0 + d_g)`
#     over `1 - stats.t.cdf(...)`: stats.t.sf does not underflow to 0 in the tail,
#     which matters for the strongest Golub genes.
#   * BH-adjust with `multipletests(p_mod, alpha=0.05, method='fdr_bh')`.

# YOUR CODE HERE (paste AI output, refine, then complete the critique above)


### Q2.b — Estimate $\hat\pi_0$ using Storey's formula

Estimate $\pi_0$ (the proportion of truly null genes) directly from the raw moderated $p$-values using Storey's adaptive estimator,

$$\hat\pi_0(\lambda) = \frac{\#\{p_i > \lambda\}}{m\,(1 - \lambda)},
\qquad \lambda = 0.5.$$

Lab 2 implemented the same recipe (cell 7). The homework version uses the moderated $p$-values from Q2.a.

*Sensitivity caveat.* The single-$\lambda$ form below is a pedagogical shortcut. Storey's original paper recommends estimating $\hat\pi_0(\lambda)$ across a grid of $\lambda$ values, smoothing the curve, and reading off the limit as $\lambda \to 1$. The fixed $\lambda = 0.5$ used here can be *unstable* on datasets like Golub where alternatives concentrate sharply near $p = 0$ — your $\hat\pi_0$ on this data may come out closer to 1 than the visible spike at zero would suggest.

**Suggested prompt** (paste-ready, with format directive):

> "Reply with a single Python code block (no prose, no .docx file). Given a numpy array `p_mod` of raw $p$-values from the moderated-$t$ analysis on the Golub data, compute Storey's $\hat\pi_0(\lambda)$ at $\lambda = 0.5$ using the formula $\hat\pi_0 = \#\{p_i > \lambda\} / (m(1-\lambda))$. Print $\hat\pi_0$ with three decimal places, then a one-line interpretation 'roughly X% of the 3,051 genes are estimated to be truly null', using the printed number. Do not invent or hallucinate numbers — only print what the formula returns."

(Notice the prompt deliberately does *not* tell the AI to cap at 1.0 — that's what the checklist tests.)

**Critique checklist (complete after pasting the AI's code):**

| Criterion | Pass / Fail / Partial | Notes |
|---|---|---|
| Did the AI report $\hat\pi_0$ greater than 1.0 (indicating the raw formula was applied without any cap)? If yes, the AI matched the prompt; if it silently capped, note that. | | |
| Did the AI use $m \cdot (1-\lambda)$ in the denominator (not $m$ alone, which would inflate to $2\hat\pi_0$)? | | |
| Did the AI use $\lambda = 0.5$ (not the default from some other library or a different $\lambda$)? | | |
| Did the AI's printed interpretation match the printed number (no hallucinated percentage)? | | |
| Did the AI fix-up any value > 1 silently? Note in the table — both behaviors are defensible if labeled. | | |


In [ ]:
# YOUR CODE HERE
# Hint:
#   lam = 0.5
#   pi0_hat = float(np.sum(p_mod > lam) / (len(p_mod) * (1 - lam)))
#   pi0_hat = min(pi0_hat, 1.0)
#   print(f"pi0_hat = {pi0_hat:.3f}; roughly {pi0_hat * 100:.1f}% of the 3,051 genes are estimated to be truly null.")


**Interpretation of $\hat\pi_0$:**



### Q2.c — P-value histogram with annotation

Make a $p$-value histogram of the moderated $p$-values from Q2.a. Annotate it with two reference lines (the uniform-null density at 1 and the estimated null bin density $\hat\pi_0$) and shade two regions of the $p$-value axis (the small-$p$ region where true alternatives concentrate, and the roughly uniform null region near 1).

**Suggested prompt** (paste-ready, with format directive):

> "Reply with a single Python code block (no prose, no .docx file). Given a numpy array `p_mod` of moderated $p$-values from the Golub data and the Storey estimate `pi0_hat`, produce a matplotlib histogram of the $p$-values using 50 bins with **density scaling** (`density=True`). Overlay a horizontal dashed reference line at the uniform-null density (height 1) labelled 'uniform-null density'. Overlay a horizontal dotted line at height `pi0_hat` labelled 'estimated null bin density (π̂₀)'. Use `ax.axvspan(0, 0.1, alpha=0.15, color='C1')` to shade the small-$p$ region (mostly true alternatives) and `ax.axvspan(0.5, 1.0, alpha=0.1, color='C0')` to shade the roughly uniform-null region. Title the figure, label both axes, and add a legend that is readable without the prompt."

**Critique checklist (complete after pasting the AI's code):**

| Criterion | Pass / Fail / Partial | Notes |
|---|---|---|
| Did the AI use `density=True` (not raw counts)? | | |
| Did the AI label the horizontal dotted line as $\hat\pi_0$ (null *bin density*, not a curve $f_0$)? | | |
| Did the AI keep the axis labels and legend self-contained (readable without the prompt)? | | |
| Did the AI use a consistent number of bins (50) so the density scale is interpretable? | | |

In [ ]:
# YOUR CODE HERE
# Sketch (50 bins, density-scaled, two reference lines, two shaded regions):
#
#   fig, ax = plt.subplots(figsize=(8, 4))
#   ax.hist(p_mod, bins=50, density=True, color='steelblue', edgecolor='white')
#   ax.axhline(1.0, color='red', linestyle='--', label='expected if all null')
#   ax.axhline(pi0_hat, color='orange', linestyle=':', label=f'estimated null density (pi0 = {pi0_hat:.2f})')
#   ax.axvspan(0, 0.1, alpha=0.15, color='C1', label='spike of true alternatives')
#   ax.axvspan(0.5, 1.0, alpha=0.10, color='C0', label='roughly uniform null region')
#   ax.set_xlabel('moderated p-value'); ax.set_ylabel('density')
#   ax.set_title('Annotated histogram of moderated p-values (Golub ALL vs AML)')
#   ax.legend()
#   plt.tight_layout(); plt.show()


**One-paragraph interpretation of (a)–(c):**

(Discuss: How many discoveries at each FDR level? What does $\hat\pi_0$ suggest about the proportion of nulls? Does the histogram shape match your expectation? What was AI-generated vs. your own?)




---
## Q3: Evaluating Analysis Output, and a Day-2 bridge (LO3)

### (a) Two specifications for the same task

Write **two** specifications for the same analysis task: *"Perform differential expression analysis on a microarray dataset comparing treatment and control groups."*

- **Specification A** (vague, likely to produce incorrect/incomplete analysis):


- **Specification B** (precise, likely to produce correct analysis on the first attempt — name the method, the test, the multiple-testing correction, and the threshold):


**What makes Specification B better?**


### (b) Verifying a citation

You read (in a paper or AI output) the claim:
> *"Gene XYZ promotes cell proliferation in breast cancer through the PI3K/AKT pathway (Smith et al., 2019)."*

Describe the steps you would take to verify this before including it in your own work.

**YOUR ANSWER:**


### (c) An error you encountered on Day 1

Describe one specific error you encountered during Day 1's labs — whether in AI-generated code, your own code, or a collaborator's analysis.

- What was the error?
- How did you catch it?
- What statistical knowledge was required to recognize it?

**Suggested AI prompt for follow-up** (paste-ready, with format directive):

> "Reply with a single Python code block (no prose, no .docx file — just runnable code I can paste into a Colab cell). Here is a snippet of analysis code: \[paste your snippet\]. Identify the most likely statistical error and write a corrected version of the snippet. If the original is correct, return the same code unchanged with a one-line comment confirming this."

**YOUR ANSWER:**


### (d) A short review — you vs. AI vs. AI-revises-itself (true multi-turn)

This is the multi-turn AI exercise required for this homework. **Turns 2 and 3 must be sent in the same chat session** so the AI can see and revise its earlier output — that is the point of "multi-turn." If you start a fresh conversation between turns, the exercise reduces to two independent prompts and you lose the value.

**Turn 1 — your review.** You receive a complete differential expression analysis from a collaborator. It uses raw two-sample $t$-tests with BH correction on a microarray dataset with 72 samples. Write a short review (3–4 sentences) describing what's adequate and what should be changed, and why.

*(Hint: connect this to Lecture 2's argument that empirical-Bayes variance shrinkage — the moderated $t$-statistic — is preferred over raw $t$-tests when $n$ is small. The [Decision Guide](https://pflahert.github.io/hd-stats-workshop/syllabus/decision-guide.html) summarizes the recommended methods.)*

**YOUR REVIEW (Turn 1):**


**Turn 2 — open an AI conversation and ask for its own review of the same scenario:**

> "Reply in three to four sentences only (no .docx file). A collaborator submitted a differential expression analysis on a microarray dataset with $n = 72$ samples using raw two-sample $t$-tests with Benjamini–Hochberg FDR correction. Write a brief critical review explaining what is adequate about this approach and what should be changed, and why."

**AI's Turn-2 review (paste here):**


**Turn 3 — in the same conversation, paste your Turn-1 review and ask the AI to compare and revise:**

> "Here is the review I wrote myself for the same scenario: \[paste your Turn-1 review verbatim\]. Compare my review to the one you wrote in your previous response. List any methodological issues I raised that you did **not** raise, and any that you raised that I did **not**. Then produce a revised version of your review that incorporates the best points from both — you may keep parts of your earlier review unchanged if you still believe them, or revise wherever I caught something you missed. Three to four sentences for the revised review. No .docx file."

**AI's Turn-3 comparison and revised review (paste here):**


**Final synthesis (your work).** Combine the best points from all three turns into a single 3–5-sentence review you would actually send to the collaborator. State explicitly which points came from you, which came from the AI's Turn-2 review, and which came from the AI's Turn-3 reconciliation. Note any case where Turn 3 changed the AI's view from Turn 2 — that is the value of staying in the same conversation context.

**FINAL SYNTHESIZED REVIEW:**


**Reflection.** What did this three-turn exchange teach you about using AI as a reviewer? Were the AI's contributions complementary to your own, or did they substantially overlap? Did the AI catch issues you missed, or did you catch issues the AI missed? Did Turn 3 surface anything Turn 2 missed?


### (e) Bridge to Day 2 — PCA preview on Golub

Before Day 2, run a quick PCA preview on the Golub matrix to prime Lecture 3 (dimension reduction) and Lecture 4 (penalized regression). The next code cell does the work; the markdown cell after it asks you to interpret the printed output in one sentence.

In [ ]:
# Q3(e) — PCA preview on Golub. Center + scale each gene, fit a 10-component
# PCA on the samples-by-genes matrix, report cumulative variance explained.
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# expr is genes x samples in the load cell above; transpose to samples x genes
X_for_pca = StandardScaler().fit_transform(expr.values.T)
pca = PCA(n_components=10).fit(X_for_pca)
cum_var = np.cumsum(pca.explained_variance_ratio_)
for k, v in enumerate(cum_var, start=1):
    print(f"PC1..PC{k}: cumulative variance explained = {v:.3f}")

print(f"\n(n, p) = {X_for_pca.shape}  --> p >> n; this is the regime Lecture 4 addresses.")


**One-sentence bridge to Day 2:** Given the cumulative variance and the $(n, p)$ shape you just printed, what does the $p \gg n$ constraint imply for the LASSO / ridge / elastic-net analyses you will run in Lab 4?

**YOUR ANSWER:**


---
## Submission

Submit your completed `homework.ipynb` (via `File → Download → Download .ipynb` in Colab) to the workshop **Harmony site** before the start of Day 2, *and* bring a printed or on-screen copy of your Q2 code output and Q3(d) synthesis to Day 2 — we will reference your answers during the Lecture 3 and Lecture 4 discussions.

### Assessment criteria

This is a formative assessment. Completeness and engagement matter more than getting every detail right. We are particularly interested in:

- Your interpretation of statistical results **in your own words** (the AI cells record AI output; the critique checklists and assessment cells record *your* evaluation).
- Your ability to identify errors in analysis code or output (Q3(c), and the AI-critique checklists throughout Q1 and Q2).
- The Q3(d) three-turn exchange: a thoughtful comparison of your review, the AI's review, and the AI's Turn-3 reconciliation.
- The Q3(e) bridge: a one-sentence connection from the Day-1 toolkit to the Day-2 methods.

> **Tip:** `File → Download → Download .ipynb` to keep a local copy of your work.
